# Part 1, extension 3a: yearly satellite images of one region (Madre de Dios, Peru)

This notebook downloads the images for the network-drawn deforestation map. Notebook 08 then runs the trained network on them.

**What the map shows.** The region is cut into 640 m squares, the same size as the training patches. For each year from 2019 to 2024, the network compares that year's image with the 2018 image, square by square. Each frame then shows what the network thinks has been cleared since 2018, so playing the frames in order shows the clearing spreading.

**Why 2018 is always the "before" image.** The network was trained on pairs of images several years apart (2018 and 2023). Comparing every year with 2018 keeps the input as close to training as possible. Comparing year to year would give the network much smaller changes than it ever saw in training.

**Same recipe as training.** The Sentinel-2 images, Cloud Score+ masking, bands, pixel size and map projection are copied from notebook 01. Only the layout changes: instead of scattered patches, a grid of squares covering one area.

**Where.** Madre de Dios was chosen by Madam Ji (decision log, Nov 6). Inside the Madre de Dios study rectangle already used in notebook 05, section 3 picks the 0.25° square (about 27 km across) with the most Hansen forest loss in 2019 to 2023. The data choose the exact spot; nobody picks it by hand.

**Checks.** Section 7 downloads a few squares again, one at a time exactly the way notebook 01 did, and stops if they differ from the big download.

## 1. Settings

In [ ]:
EE_PROJECT = "earth-01-final"

BEFORE_YEAR = 2018
AFTER_YEARS = [2019, 2020, 2021, 2022, 2023, 2024]   # one map frame per year

# The Madre de Dios study rectangle from notebook 05 (MAAP #124 hotspots), [west, south, east, north]
SEARCH_BOX = [-71.5, -13.5, -68.7, -11.0]
WINDOW_DEG = 0.25        # size of the mapped square, about 27 km
COARSE_DEG = 0.01        # resolution of the search for the busiest square, about 1 km
EXPECTED_WINDOW = [-70.06, -12.9, -69.81, -12.65]   # result of notebook 07c (Nov 6), checked again below

# Copied from notebook 01, unchanged
PATCH_PX = 64
PIXEL_M = 10
BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]
CLOUD_SCORE_MIN = 0.60
TREECOVER_MIN = 30
SEED = 42

CHUNK_TILES = 8          # squares per side in one download request (512 x 512 pixels)

OUT_DIR = "../data/raw/part1_map"
GRID_JSON = "../data/cleaned/part1_map_grid.json"
TILES_CSV = "../data/cleaned/part1_map_tiles.csv"
FIG_DIR = "../outputs"

assert max(AFTER_YEARS) <= 2024 and BEFORE_YEAR < min(AFTER_YEARS)

## 2. Connect to Earth Engine

In [ ]:
import os
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import ee
import numpy as np
import pandas as pd
from numpy.lib import recfunctions as rfn
from pyproj import Transformer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

ee.Authenticate()
ee.Initialize(project=EE_PROJECT, opt_url="https://earthengine-highvolume.googleapis.com")
ee.data.setDeadline(180_000)

# Cached chunks are named by window, so chunks from another window can never be reused by mistake
CACHE_DIR = os.path.join(OUT_DIR, "chunk_cache_" + "_".join(f"{v:.2f}" for v in EXPECTED_WINDOW))
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(GRID_JSON), exist_ok=True)

## 3. Find the busiest 27 km square in Madre de Dios

Hansen forest loss in 2019 to 2023 is averaged onto a 1 km grid over the study rectangle. A 0.25° window then slides across it, and the window with the highest share of lost forest is kept. Only windows fully inside Peru (FAO GAUL outline) count.

In [ ]:
hansen = ee.Image("UMD/hansen/global_forest_change_2023_v1_11")
treecover = hansen.select("treecover2000").unmask(0)
lossyear = hansen.select("lossyear").unmask(0)
datamask = hansen.select("datamask").unmask(0)
forest2000 = treecover.gte(TREECOVER_MIN).And(datamask.eq(1))

peru = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq("ADM0_NAME", "Peru")).geometry()
recent = forest2000.And(lossyear.gte(19)).And(lossyear.lte(23))
peru_fc = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq("ADM0_NAME", "Peru"))
# Peru is painted onto an empty image. The first version clipped a constant image to Peru instead,
# and 11% of those cells came back infinite, so the east of the rectangle was never searched
# (found and fixed with notebooks 07b and 07c, Nov 6).
search = (recent.toFloat().reduceResolution(ee.Reducer.mean(), maxPixels=4096).rename("recent")
          .addBands(ee.Image(0).paint(peru_fc, 1).toFloat().rename("peru")))

w, s, e, n = SEARCH_BOX
W, H = int(round((e - w) / COARSE_DEG)), int(round((n - s) / COARSE_DEG))
arr = ee.data.computePixels({
    "expression": search, "fileFormat": "NUMPY_NDARRAY",
    "grid": {"dimensions": {"width": W, "height": H},
             "affineTransform": {"scaleX": COARSE_DEG, "shearX": 0, "translateX": w,
                                 "shearY": 0, "scaleY": -COARSE_DEG, "translateY": n},
             "crsCode": "EPSG:4326"}})
recent_map, peru_map = arr["recent"].astype(float), arr["peru"].astype(float)
assert np.isfinite(recent_map).all() and np.isfinite(peru_map).all(), "the search grid has broken cells; stop here"
print(f"Search grid {H} x {W} cells; forest lost 2019-2023 over the whole rectangle: {recent_map.mean():.2%}")

def window_sums(a, k):
    c = np.pad(a, ((1, 0), (1, 0))).cumsum(0).cumsum(1)
    return c[k:, k:] - c[:-k, k:] - c[k:, :-k] + c[:-k, :-k]   # sum of every k x k window

k = int(round(WINDOW_DEG / COARSE_DEG))
loss_share = window_sums(recent_map, k) / k**2
inside = window_sums(peru_map, k) >= k**2 - 0.5
assert inside.any(), "no window lies fully inside Peru"
score = np.where(inside, loss_share, -1)
i, j = np.unravel_index(np.argmax(score), score.shape)
WINDOW = [w + j * COARSE_DEG, n - (i + k) * COARSE_DEG, w + (j + k) * COARSE_DEG, n - i * COARSE_DEG]
WINDOW = [float(round(v, 4)) for v in WINDOW]
# Notebook 07c found this window over the whole rectangle with a checked Peru layer; the two must agree
assert np.allclose(WINDOW, EXPECTED_WINDOW, atol=1e-6), f"search found {WINDOW}, notebook 07c found {EXPECTED_WINDOW}"
print(f"Chosen window [west, south, east, north]: {WINDOW}")
print(f"Forest lost there 2019-2023: {score[i, j]:.1%} of the area "
      f"(median window inside Peru: {np.median(loss_share[inside]):.2%})")

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(recent_map, extent=[w, e, s, n], cmap="Reds", vmax=np.percentile(recent_map, 99.5))
ax.contour(np.flipud(peru_map), levels=[0.5], extent=[w, e, s, n], colors="k", linewidths=0.6, origin="lower")
x0_, y0_, x1_, y1_ = WINDOW
ax.plot([x0_, x1_, x1_, x0_, x0_], [y0_, y0_, y1_, y1_, y0_], color="blue", lw=1.5)
ax.set_title("Forest lost 2019-2023 (Hansen), chosen window in blue", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "part1_map_window.png"), dpi=150)
plt.show()

## 4. Lay out the grid of 640 m squares

Same projection as notebook 01 (EPSG:3857), with the pixel size stretched by 1/cos(latitude) so each pixel is 10 m on the ground. Notebook 01 used each patch's own latitude; here one latitude (the window's centre) is used for the whole grid. Across 0.25° the difference is about 0.1%, far below one pixel per square.

In [ ]:
to_merc = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
to_geo = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

lat_c = (WINDOW[1] + WINDOW[3]) / 2
px = PIXEL_M / np.cos(np.radians(lat_c))
tile_m = PATCH_PX * px
x0, y0 = to_merc.transform(WINDOW[0], WINDOW[3])     # top-left corner
x1, y1 = to_merc.transform(WINDOW[2], WINDOW[1])     # bottom-right corner
ROWS, COLS = int((y0 - y1) // tile_m), int((x1 - x0) // tile_m)
print(f"{ROWS} x {COLS} = {ROWS * COLS} squares of 640 m, {ROWS * PATCH_PX} x {COLS * PATCH_PX} pixels")

grid = {"window": WINDOW, "lat_centre": lat_c, "pixel_size_merc": px, "x0": x0, "y0": y0,
        "rows": ROWS, "cols": COLS, "tile_px": PATCH_PX, "before_year": BEFORE_YEAR,
        "after_years": AFTER_YEARS, "crs": "EPSG:3857", "hansen": "UMD/hansen/global_forest_change_2023_v1_11"}
with open(GRID_JSON, "w") as f:
    json.dump(grid, f, indent=2)

chunks = [(r, c, min(CHUNK_TILES, ROWS - r), min(CHUNK_TILES, COLS - c))
          for r in range(0, ROWS, CHUNK_TILES) for c in range(0, COLS, CHUNK_TILES)]
print(f"{len(chunks)} download chunks per layer")

## 5. Download the yearly images and the Hansen layers

`composite()` is copied from notebook 01. Each year is downloaded on its own with its own valid-data band. Notebook 08 joins 2018 with each later year exactly as notebook 01 joined its two years: the six 2018 bands, then the six later-year bands, and a pixel counts as valid only if both years have data. Each chunk is saved as soon as it arrives, so an interrupted run picks up where it stopped.

In [ ]:
s2 = ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
cs_plus = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")

def composite(year, region):          # copied from notebook 01
    col = (s2.filterBounds(region)
             .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
             .linkCollection(cs_plus, ["cs_cdf"])
             .map(lambda img: img.updateMask(img.select("cs_cdf").gte(CLOUD_SCORE_MIN))))
    return col.select(BANDS).median()

def year_image(year, region):
    img = composite(year, region)
    valid = img.select(0).mask().rename("valid")     # same valid test as notebook 01
    return img.unmask(0).toUint16().addBands(valid.unmask(0).toUint16())

hansen_layers = (treecover.addBands(lossyear).addBands(datamask)
                 .rename(["treecover2000", "lossyear", "datamask"]).toUint16())

def request(expression, r, c, h, w_):
    return {"expression": expression, "fileFormat": "NUMPY_NDARRAY",
            "grid": {"dimensions": {"width": w_ * PATCH_PX, "height": h * PATCH_PX},
                     "affineTransform": {"scaleX": px, "shearX": 0, "translateX": x0 + c * tile_m,
                                         "shearY": 0, "scaleY": -px, "translateY": y0 - r * tile_m},
                     "crsCode": "EPSG:3857"}}

def chunk_region(r, c, h, w_):
    lon_a, lat_a = to_geo.transform(x0 + c * tile_m, y0 - r * tile_m)
    lon_b, lat_b = to_geo.transform(x0 + (c + w_) * tile_m, y0 - (r + h) * tile_m)
    return ee.Geometry.Rectangle([lon_a, lat_b, lon_b, lat_a]).buffer(1000).bounds()

def fetch_chunk(key, r, c, h, w_, retries=5):
    path = os.path.join(CACHE_DIR, f"{key}_r{r:03d}_c{c:03d}.npy")
    if os.path.exists(path):
        return key, r, c, np.load(path)
    expr = hansen_layers if key == "hansen" else year_image(int(key), chunk_region(r, c, h, w_))
    for attempt in range(retries):
        try:
            arr = ee.data.computePixels(request(expr, r, c, h, w_))
            out = rfn.structured_to_unstructured(arr).transpose(2, 0, 1).astype(np.uint16)
            np.save(path, out)
            return key, r, c, out
        except Exception as err:
            if attempt == retries - 1:
                print(f"{key} chunk ({r},{c}) failed: {str(err)[:120]}")
                return key, r, c, None
            time.sleep(2 ** attempt)

layers = ["hansen", str(BEFORE_YEAR)] + [str(y) for y in AFTER_YEARS]
n_bands = {"hansen": 3}
mosaics = {}
for key in layers:
    nb = n_bands.get(key, len(BANDS) + 1)
    mos = np.zeros((nb, ROWS * PATCH_PX, COLS * PATCH_PX), dtype=np.uint16)
    failed = 0
    with ThreadPoolExecutor(max_workers=8) as pool:
        jobs = [pool.submit(fetch_chunk, key, *ch) for ch in chunks]
        for job in tqdm(as_completed(jobs), total=len(jobs), desc=key):
            _, r, c, out = job.result()
            if out is None:
                failed += 1
                continue
            assert out.shape[0] == nb, f"{key}: expected {nb} bands, got {out.shape[0]}"
            mos[:, r * PATCH_PX:r * PATCH_PX + out.shape[1], c * PATCH_PX:c * PATCH_PX + out.shape[2]] = out
    assert failed == 0, f"{failed} chunks of {key} failed; run this cell again (finished chunks are cached)"
    np.save(os.path.join(OUT_DIR, f"mosaic_{key}.npy"), mos)
    if key in ("hansen", str(BEFORE_YEAR), str(AFTER_YEARS[-1])):   # kept in memory for sections 6 and 7
        mosaics[key] = mos
    if key != "hansen":
        print(f"  {key}: clear-sky pixels {mos[-1].mean():.1%}")

## 6. Hansen numbers for every square

For every 640 m square, the share of its area that lost forest in each year, and the other numbers notebook 01's label rules use. Notebook 08 scores the network against these with the same rules as training.

In [ ]:
def tile_stats(hm, rows, cols, T=PATCH_PX):
    tc, ly, dm = hm[0].astype(np.int16), hm[1].astype(np.int16), hm[2]
    forest = (tc >= TREECOVER_MIN) & (dm == 1)            # forest in 2000, as in notebook 01
    def per_tile(m):
        return m.reshape(rows, T, cols, T).mean(axis=(1, 3))
    t = {"treecover_mean": per_tile(tc.astype(float)),
         "land_frac": per_tile(dm == 1),
         "lost_before_frac": per_tile(forest & (ly > 0) & (ly < 19)),       # lost 2001-2018
         "forest2018_frac": per_tile(forest & ~((ly > 0) & (ly < 19)))}     # forest left at end of 2018
    for yr in range(2019, 2024):
        t[f"lostF_{yr}"] = per_tile(forest & (ly == yr - 2000))           # forest lost in that year
    for yr in range(2018, 2024):
        t[f"lostA_upto_{yr}"] = per_tile((ly > 0) & (ly <= yr - 2000))    # any loss 2001 up to that year
    rr, cc = np.meshgrid(np.arange(rows), np.arange(cols), indexing="ij")
    df = pd.DataFrame({"row": rr.ravel(), "col": cc.ravel()})
    for name, v in t.items():
        df[name] = v.ravel()
    return df

hm = mosaics["hansen"]
tiles = tile_stats(hm, ROWS, COLS)
lon, lat = to_geo.transform(x0 + (tiles["col"].to_numpy() + 0.5) * tile_m,
                            y0 - (tiles["row"].to_numpy() + 0.5) * tile_m)
tiles.insert(0, "tile_id", [f"m{r:03d}_{c:03d}" for r, c in zip(tiles["row"], tiles["col"])])
tiles["lon"], tiles["lat"] = lon, lat
tiles.to_csv(TILES_CSV, index=False)

assert tiles["lostF_2019"].between(0, 1).all() and len(tiles) == ROWS * COLS
print(f"{len(tiles)} squares saved to {TILES_CSV}")
print(f"Mostly forest at the end of 2018 (at least half): {(tiles.forest2018_frac >= 0.5).mean():.0%} of squares")
print("Share of the window's area that lost forest, by year (Hansen):")
print(tiles[[f"lostF_{y}" for y in range(2019, 2024)]].mean().map("{:.2%}".format).to_string())

## 7. Check: the big download matches notebook 01's way of downloading

Five squares are picked at random (fixed seed) and downloaded again one at a time, exactly the way notebook 01 fetched each patch (its own 64 × 64 request, with scenes chosen around that patch only). They must match the same squares cut from the big download. This catches any mistake in how the chunks were placed or turned.

In [ ]:
rng = np.random.default_rng(SEED)
check_tiles = tiles.sample(5, random_state=SEED)
check_years = [BEFORE_YEAR, AFTER_YEARS[-1]]
half_m = PATCH_PX * PIXEL_M / 2
results = []
for t in check_tiles.itertuples():
    footprint = ee.Geometry.Point([t.lon, t.lat]).buffer(2 * half_m).bounds()   # as in notebook 01
    for yr in check_years:
        single = ee.data.computePixels(request(year_image(yr, footprint), t.row, t.col, 1, 1))
        single = rfn.structured_to_unstructured(single).transpose(2, 0, 1).astype(np.uint16)
        big = mosaics[str(yr)][:, t.row * PATCH_PX:(t.row + 1) * PATCH_PX, t.col * PATCH_PX:(t.col + 1) * PATCH_PX]
        same = (single == big).mean()
        results.append({"tile": t.tile_id, "year": yr, "identical_share": same,
                        "max_difference": int(np.abs(single.astype(int) - big.astype(int)).max())})
check = pd.DataFrame(results)
print(check.to_string(index=False))
assert (check["identical_share"] >= 0.99).all(), "the big download does not match single downloads; stop here"
print("Check passed: the big download matches notebook 01's single-patch download")
with open(os.path.join(FIG_DIR, "part1_map_download_check.json"), "w") as f:
    json.dump({"window": WINDOW, "rows": ROWS, "cols": COLS, "check": results}, f, indent=2, default=float)

## Sources and credits

- Hansen, M. C. et al. (2013). High-Resolution Global Maps of 21st-Century Forest Cover Change. *Science* 342, 850–853. Data: University of Maryland, CC BY 4.0.
- Contains modified Copernicus Sentinel data (2018–2024).
- Pasquarella, V. J. et al. (2023). Cloud Score+ for Sentinel-2.
- FAO Global Administrative Unit Layers (GAUL) 2015.
- Study rectangle: MAAP #124, Deforestation Hotspots 2020 in the Peruvian Amazon (Amazon Conservation).